# 6-1 · 대출 신청은 어떤 경로로 끝나는가 — 이벤트 로그와 상태 지도

**⏱ 60분** · 6주차 교재 1/2 — 이벤트 로그 읽기, 종료 시점, 상태 지도 → (6-2) 구간 시간과 미비 반복, 무엇을 먼저 시험할까

## 🎯 이번 주의 질문

대출 신청은 생성된 뒤 서류 확인과 검증을 거쳐 확정·취소·거절로 끝납니다. 같은 상태를 여러 번 거치는 신청도 있습니다. 심사 기간을 줄이려는 운영팀은 어느 단계부터 자세히 조사하고 개선을 시험할지 정해야 합니다.

> **대출 신청 과정에서 오래 걸리고 반복되는 구간은 어디이며, 어느 구간부터 개선을 시험할까요?**

시스템에 남은 기록으로 과정의 흐름과 반복을 찾고 시간을 비교하는 분석을 **프로세스 마이닝**이라고 합니다. 은행·보험·병원처럼 한 건이 여러 단계를 거치는 업무에서 개선 후보를 찾을 때 씁니다.

이번 시간에는 기록을 신청 단위로 읽고, 신청마다 어디까지 잴지 정한 뒤, 상태가 어떤 순서로 이어지는지 **지도**로 그립니다. 구간별 시간과 반복은 6-2에서 봅니다.

**데이터:** 네덜란드 금융기관이 공개한 [BPI Challenge 2017](https://doi.org/10.4121/uuid:5f3067df-f10b-45da-b98b-86ae4c7a310b) 대출 신청 기록입니다. 2016년 한 해 동안 생성된 신청의 처리 과정이 2017년 2월 1일까지 기록되어 있습니다.

**이번 시간에 반복할 패턴:** 신청별로 묶기 `groupby("case_id")` · 다음 행 가져오기 `shift(-1)` · 쌍으로 세기 `groupby([열1, 열2]).size()`

### 셀 표시 읽는 법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행하고 **결과만** 읽습니다. 코드를 외울 필요는 없습니다 |
| ✍️ | 직접 입력 | 보여 준 코드를 **손으로 타이핑**해서 실행합니다. 이번 주에 반복하는 핵심 패턴입니다 |
| 📝 | 과제 | 앞에서 입력한 코드를 **조금 바꿔서** 스스로 풉니다 |
| 💬 · ✅ | 결과 해설 · 체크포인트 | 숫자를 읽는 법, 내 결과가 맞는지 확인할 값 |

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 대출 신청 이벤트 로그를 표 log로 읽습니다. (20~30초)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
log = load("loan_process")
print("준비 완료!")

✅ `준비 완료!`가 나오면 됩니다.

---
## 1. 이벤트 로그의 한 행은 무엇인가

표의 크기와 앞부분부터 봅니다.

```python
print(log.shape)
log.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — 1,202,267행 × 7열입니다. 앞의 다섯 행은 모두 같은 신청 번호 `Application_1000086665`이고, 활동과 시각만 다릅니다.

- 대출 신청 하나를 **케이스**(case), 그 신청에 일어난 활동 하나의 기록을 **이벤트**(event)라고 합니다. **한 행 = 이벤트 하나**이고, 신청 하나가 여러 행으로 나타납니다.
- 은행은 신청의 상태가 바뀌거나 직원의 작업이 시작·중단될 때 이벤트를 남겼습니다. 고객이 실제로 서류를 보낸 시각과 은행이 기록한 시각이 같다는 보장은 없습니다.

| 열 | 기록된 내용 |
|---|---|
| `case_id` | 신청 번호. 같은 신청은 여러 행으로 나타난다 |
| `activity` | 활동 이름. `A_`는 신청 상태, `O_`는 오퍼, `W_`는 직원 작업 항목 |
| `lifecycle` | 작업의 생성·착수·중단·재개·완료 등 기록 시점 |
| `timestamp` | 이벤트가 기록된 시각 |
| `resource` | 이벤트를 남긴 담당자 또는 시스템 |
| `RequestedAmount`, `LoanGoal` | 신청 금액(유로), 신청 목적 |

오퍼는 은행이 고객에게 제시한 대출 조건이며, 한 신청에 여러 개가 만들어질 수 있습니다.

신청이 몇 건인지 셉니다. `.nunique()`는 서로 다른 값의 개수입니다.

```python
log["case_id"].nunique()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ **체크포인트** — `31509`이면 맞습니다.

💬 신청 31,509건에 이벤트 120만 개, 신청 하나에 평균 38개쯤의 기록이 있습니다. 행 수와 신청 수를 헷갈리지 않아야 합니다.

---
## 2. 신청 상태에는 어떤 것이 있나

과정의 흐름은 신청 **상태**(`A_`로 시작하는 활동)로 읽습니다. `.str.startswith("A_")`는 글자가 `A_`로 시작하면 참입니다. 조건 필터로 상태 행만 남기고 활동별로 셉니다.

```python
a_events = log[log["activity"].str.startswith("A_")]
a_events["activity"].value_counts()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — 상태는 10가지입니다. 이 장에서는 다음과 같이 읽습니다.

| 상태 | 뜻 | 기록 수 |
|---|---|---:|
| A_Create Application | 신청 생성 | 31,509 |
| A_Submitted | 고객이 신청을 제출한 기록 | 20,423 |
| A_Concept | 신청 내용을 작성·확인하는 단계 | 31,509 |
| A_Accepted | 오퍼를 준비할 수 있는 단계로 진행 | 31,509 |
| A_Complete | 신청서 작성 완료 | 31,362 |
| A_Validating | 검증 중 | 38,816 |
| A_Incomplete | 서류 미비가 기록됨 | 23,055 |
| A_Pending | 대출 확정 단계에 도달 | 17,228 |
| A_Cancelled | 취소 | 10,431 |
| A_Denied | 거절 | 3,753 |

- 이 장에서 **확정**은 `A_Pending` 도달을 뜻합니다. 실제 대출금 지급까지 잰다는 뜻은 아니고, `A_Accepted`와도 다릅니다.
- 검증(38,816)이 신청 수(31,509)보다 많습니다. 같은 신청이 검증을 여러 번 거쳤다는 뜻입니다.

---
## 3. 신청 하나를 따라가 본다

신청 하나의 기록을 시각 순서로 읽으려면 표를 **신청 번호 → 시각** 순으로 정렬해야 합니다. 같은 시각의 기록은 원래 순서를 지키도록 원래 행 번호(`event_order`)를 셋째 기준으로 둡니다. 정렬 코드는 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 신청 번호 → 시각 → 원래 순서로 정렬한 표 ordered
ordered = log.reset_index().rename(columns={"index": "event_order"})
ordered = ordered.sort_values(["case_id", "timestamp", "event_order"])

예시 신청 `Application_1005465952`의 상태 기록만 봅니다. 신청 번호로 한 번, 상태로 한 번 **조건 필터**를 겁니다.

```python
one = ordered[ordered["case_id"] == "Application_1005465952"]
one[one["activity"].str.startswith("A_")][["activity", "timestamp"]]
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — 7월 27일 생성되어 8월 5일 확정(`A_Pending`)됐습니다.
- 작성 완료(7/27) 뒤 검증(8/1)까지 5일, 검증 중에 미비가 기록(8/5)되고 다시 검증으로 돌아간 뒤 확정됐습니다.
- 상태와 상태 사이에는 전화·서류 전달·직원 처리 같은 여러 일이 있었을 수 있습니다. 기록으로 알 수 있는 것은 **다음 상태가 기록될 때까지 얼마나 걸렸는가**입니다. 이 경과 시간을 모든 신청에서 모으면 어디가 긴지 알 수 있습니다.

---
## 4. 신청마다 어디까지 잴 것인가

신청마다 기간을 같은 기준으로 재려면 시작과 끝을 정해야 합니다.

| 항목 | 정의 |
|---|---|
| 시작 | 신청 생성(`A_Create Application`) 시각 |
| 종료 | 확정·취소·거절(`A_Pending`·`A_Cancelled`·`A_Denied`) 중 **처음** 기록된 시각 |
| 흐름 시간 | 시작부터 종료까지의 달력상 경과 일수(밤과 주말 포함) |

종료 뒤에 작업 항목을 닫은 기록이 남더라도 심사 기간에 더하지 않습니다. 종료가 기록되지 않은 신청은 기간을 확정할 수 없으므로 따로 셉니다.

**신청별로 묶기** — 이번 주에 가장 많이 쓰는 패턴입니다. 생성 기록만 남긴 뒤 `case_id`로 묶어 시각의 최솟값을 구하면 신청마다 시작 시각이 나옵니다.

```python
starts = ordered[ordered["activity"] == "A_Create Application"].groupby("case_id")["timestamp"].min()
starts.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


신청마다 **처음 나온** 종료 상태를 찾아 시작 표 `cases`에 종료 시각과 결과를 붙입니다. `isin`으로 세 종료 상태를 고르고, 정렬된 표에서 신청마다 첫 행만 남깁니다(`drop_duplicates`). 신청 번호를 인덱스로 두면 번호끼리 맞춰 붙습니다. 이 준비 코드는 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 신청마다 첫 종료 상태와 시각을 붙인 표 cases
final = ordered[ordered["activity"].isin(["A_Pending", "A_Cancelled", "A_Denied"])]
final = final.drop_duplicates("case_id").set_index("case_id")
cases = starts.to_frame("start")
cases["end"] = final["timestamp"]
cases["outcome"] = final["activity"]

흐름 시간을 일 단위로 계산합니다. 두 시각의 차이에 `.dt.total_seconds()`를 쓰면 초가 나오고, 86,400(하루의 초)으로 나누면 소수점까지 있는 일수가 됩니다.

```python
cases["days"] = (cases["end"] - cases["start"]).dt.total_seconds() / 86400
cases.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 첫 신청은 32.6일 만에 취소, 둘째는 8.0일 만에 확정됐습니다. 종료가 없는 신청은 `end`와 `days`가 빈 값(`NaT`, `NaN`)입니다.

종료가 확인된 신청과 아닌 신청을 나눕니다. `.isna().sum()`은 빈 값의 개수, `.dropna(subset=[열])`은 그 열이 빈 행을 뺍니다.

```python
unclosed_count = cases["end"].isna().sum()
closed = cases.dropna(subset=["end"]).copy()
unclosed_count, len(closed)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ **체크포인트** — `(98, 31411)`이면 맞습니다.

💬 종료 미확인 98건, 종료 확인 31,411건입니다. 이후 비교는 종료 확인 건만 대상으로 합니다. 진행 중인 신청 전체의 예상 기간으로 읽지는 않습니다.

종료 결과마다 건수와 흐름 시간을 봅니다. **신청별로 묶기**와 같은 `groupby`인데, 이번에는 결과(`outcome`)로 묶고 `.agg([...])`로 건수·평균·중앙값을 한 번에 구합니다.

```python
flow = closed.groupby("outcome")["days"].agg(["size", "mean", "median"])
flow.round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ **체크포인트** — `A_Pending`의 size 17228, mean 18.11이면 맞습니다.

💬 **결과 해설**
- 확정 17,228건은 평균 18.11일, 거절 3,752건은 16.74일, 취소 10,431건은 29.86일입니다.
- 취소의 중앙값은 31.61일로, 한 달 부근에 몰려 있습니다. 그 이유와 취소 규칙이 언제부터 세는지는 운영 담당자에게 확인할 일입니다.
- 결과에 따라 기간이 다르므로, 6-2에서 미비 횟수에 따른 기간을 비교할 때는 **확정 신청끼리** 비교합니다.

---
## 5. 상태는 어디서 갈라지고 되돌아오나

여러 신청의 상태 순서를 모으면 공통 경로와 반복 경로가 보입니다. "상태 A 바로 다음에 상태 B"가 몇 번 나왔는지 세어 지도로 그립니다. 지도에는 상태(`A_`)만 쓰고, 직원 작업과 오퍼는 상태 사이에 있는 일로 둡니다.

먼저 지도에 쓸 기록을 고릅니다. 종료가 확인된 신청의 상태 기록 중, **첫 종료 시각까지**만 남깁니다. `map`은 신청 번호마다 `closed`의 종료 시각을 찾아 그 신청의 모든 행에 붙입니다. 준비 코드이므로 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 종료 확인 신청의 상태 기록을 첫 종료까지 남긴 표 states
states = ordered[ordered["activity"].str.startswith("A_")].copy()
states["end"] = states["case_id"].map(closed["end"])
states = states[states["end"].notna() & (states["timestamp"] <= states["end"])].copy()
states[["case_id", "activity", "timestamp"]].head()

### 다음 상태 붙이기 — 작은 예부터

각 행에 **바로 다음 상태**를 붙이려면 `shift(-1)`을 씁니다. 다음 행의 값을 현재 행으로 가져옵니다. 다만 신청이 바뀌는 곳에서 다음 신청의 첫 상태가 붙으면 안 되므로 **신청별로 묶어서** 옮깁니다. 신청 두 개짜리 작은 표로 확인합니다.

```python
example = pd.DataFrame({"case": ["가", "가", "가", "나", "나"],
                        "state": ["생성", "미비", "확정", "생성", "취소"]})
example["next"] = example.groupby("case")["state"].shift(-1)
example
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 신청 "가"의 마지막 상태(확정)에는 다음 상태가 없어 `NaN`이 붙었습니다. 신청 "나"의 "생성"이 "가"의 "확정" 뒤에 붙지 않았습니다.

### 전체 기록에서 상태 쌍 세기

같은 방법으로 `states`에 다음 상태 `next`를 붙이고, 다음 상태가 없는 마지막 행을 뺀 표를 `edges`로 둡니다. 그리고 **(현재 상태, 다음 상태) 쌍으로 묶어** 개수를 셉니다. 두 열로 묶을 때는 `groupby([열1, 열2])`입니다.

```python
states["next"] = states.groupby("case_id")["activity"].shift(-1)
edges = states.dropna(subset=["next"]).copy()
pairs = edges.groupby(["activity", "next"]).size().sort_values(ascending=False)
pairs
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ **체크포인트** — 첫 줄이 `A_Concept  A_Accepted  31411`이고, 줄이 모두 20개(상태 쌍 20가지)면 맞습니다.

💬 **결과 해설**
- 작성 → 수락(31,411)은 모든 종료 신청이 거쳤습니다.
- 검증 → 미비 22,965번, 미비 → 검증 16,894번. 검증과 미비 사이를 오가는 **반복**이 큰 흐름입니다.
- 생성 뒤 제출 기록을 거치는 경로(20,339)와 곧바로 작성 단계로 가는 경로(11,072)가 모두 있습니다.
- 한 신청에서 검증과 미비가 반복되면 같은 쌍에 여러 번 더해집니다. **쌍의 횟수를 신청 건수로 읽지 않습니다.**

상태 쌍을 그림으로 그립니다. 상태는 상자, 이어짐은 화살표, 화살표 위 숫자는 횟수입니다. 그림 도구(graphviz) 코드는 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 상태 지도 (관측된 20쌍 모두)
import graphviz
process = graphviz.Digraph(graph_attr={"rankdir": "LR"}, node_attr={"shape": "box"})
for (before, after), count in pairs.items():
    process.edge(before, after, label=str(count))
process

💬 검증 ↔ 미비의 반복, 그리고 미비에서 바로 확정·취소로 가는 경로를 함께 읽습니다. 이 지도는 **관측된** 상태의 이어짐이며, 업무를 반드시 이 순서로 해야 한다는 규정도는 아닙니다.

---
## 6. 실습 과제 (10분)

### 📝 과제 1 · 미비 뒤에는 어디로 가나

5절의 `pairs`에서 현재 상태가 미비(`A_Incomplete`)인 쌍만 골라 보세요. 미비가 기록된 뒤 반드시 검증을 다시 거쳐야 확정되나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: pairs는 (현재 상태, 다음 상태) 두 단계 인덱스를 가진 Series입니다. .loc에 현재 상태 이름 하나만 주면 그 상태에서 시작하는 쌍만 나옵니다.


✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 취소로 끝난 신청 하나 따라가기

3절과 같은 방법으로 신청 `Application_1000086665`의 상태 기록을 시각 순서로 보세요. 4절에서 이 신청은 32.6일 만에 취소됐습니다. 어느 상태 뒤에 가장 오래 머물렀나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 3절의 두 줄에서 신청 번호만 바꿉니다.


✏️ **나의 답:**

### 📝 과제 3 · 생각해 보기

같은 신청에서 "검증 → 미비"가 두 번 나타나면, 5절의 쌍 횟수와 그 구간을 거친 신청 수는 어떻게 다를까요? 지도의 숫자를 "몇 건의 신청"으로 읽으면 무엇이 문제인지 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 이벤트 로그 | 한 행 = 이벤트 하나. 신청 31,509건에 이벤트 1,202,267개 |
| 신청 상태 | `A_`로 시작하는 10가지. 확정은 `A_Pending` 도달 |
| 종료 시점 | 생성부터 첫 확정·취소·거절까지. 종료 확인 31,411건, 미확인 98건 |
| 결과별 기간 | 확정 17,228건 평균 18.11일 · 거절 16.74일 · 취소 29.86일(중앙값 31.61일) |
| 상태 지도 | 연속한 두 상태 20가지. 검증 ↔ 미비 반복이 큰 흐름. 쌍 횟수 ≠ 신청 수 |
| 반복한 패턴 | `groupby("case_id")` · `groupby(...).shift(-1)` · `groupby([열1, 열2]).size()` |

다음: **6-2 · 어느 구간에서 오래 걸리고 반복되는가 — 구간 시간과 미비 반복**